# Default factories

Já vimos isso antes, mas vamos garantir que entendemos como lidar com defaults mutáveis — ou, de forma mais geral, como criar defaults que
exigem chamar uma função.

Primeiro, Python puro.

Quando definimos valores default numa função:

In [1]:
def my_func(a=1, b=2):
    pass

`my_func` é compilada uma vez (quando o módulo que a contém é carregado pela primeira vez a cada execução da aplicação).

Funções são objetos e têm estado. Em particular, guardam o código que executam quando chamadas — mas também guardam os valores default.

Ou seja, os defaults são calculados **uma vez**, quando a função é criada. A partir daí, a cada chamada, os defaults vêm do estado da
função — não são recalculados.

Entender isso é muito importante, porque leva a bugs.

Vejamos esta função, que escreve um texto no console prefixado por um datetime informado ou, se não for informado, pelo datetime atual.

Você pode se sentir tentado a fazer assim:

In [2]:
from datetime import datetime, UTC
import time

print(f"datetime.now(UTC).isoformat(): {datetime.now(UTC).isoformat()}")
def log(text: str, dt: datetime = datetime.now(UTC)):
    print(f"{dt.isoformat()}: {text}")

datetime.now(UTC).isoformat(): 2026-11-11T11:25:47.677484+00:00


Testando:

In [3]:
log("line 1", dt=datetime(2020, 1, 1, 15, 0, 0))

2020-01-01T15:00:00: line 1


E sem informar `dt`:

In [4]:
log("line 2")

2026-11-11T11:25:47.677593+00:00: line 2


Esperamos alguns segundos e testamos de novo (o `sleep` deixa o exemplo reproduzível sem depender de quanto tempo você leva entre as células):

In [5]:
time.sleep(2)
log("line 3")

2026-11-11T11:25:47.677593+00:00: line 3


Notou algo estranho? O horário **não mudou**!

Isso porque o default foi calculado quando a função foi criada (quando executamos a célula com o `def`).

Depois disso, a cada chamada, o valor vem do estado da função — por isso vemos o mesmo valor sempre.

É o mesmo motivo pelo qual um default mutável num argumento de função causa bugs — sem entrar em detalhes, você provavelmente **nunca**
deve fazer isto:

In [6]:
def my_func(items=[]):
    pass

A solução, para este caso e para o anterior, é usar `None` como default e, **dentro** da função (código que roda a cada chamada), criar a
lista vazia ou obter o datetime atual se o argumento for `None`.

O próximo problema não é exatamente o mesmo, mas é parecido.

Nos modelos Pydantic, lembre que usamos tranquilamente código assim:

In [7]:
from pydantic import BaseModel

In [8]:
class Model(BaseModel):
    elements: list[int] = []

Numa classe comum, isso não seria boa ideia:

In [9]:
class Model:
    elements: list[int] = []

In [10]:
m = Model()
print(f"m.elements: {m.elements!r}")

m.elements: []


In [11]:
m.elements.append(1)
print(f"m.elements: {m.elements!r}")

m.elements: [1]


In [12]:
m2 = Model()
print(f"m2.elements: {m2.elements!r}")

m2.elements: [1]


Você deve ter notado que aqui lidamos com um atributo **de classe**, não de instância. Bem observado!

Mas temos exatamente o mesmo problema com dataclasses, em que usamos atributos de classe para definir atributos de instância. Aliás, as
dataclasses nem deixam fazer isso:

In [13]:
from dataclasses import dataclass

In [14]:
try:
    @dataclass
    class Model:
        elements: list[int] = []
except ValueError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValueError: mutable default <class 'list'> for field elements is not allowed: use default_factory


Repare na dica da exceção: `use default_factory`. Voltamos a isso em seguida.

O Pydantic, porém, **permite** defaults mutáveis, como acabamos de ver. Por que funcionam sem causar problemas?

In [15]:
class Model(BaseModel):
    elements: list[int] = []

O segredo é que o Pydantic **copia** o default a cada nova instância (para defaults não-hashable, como listas e dicts), então cada instância
recebe a sua própria lista.

> 📝 **Nota de revisão:** o original dizia que o Pydantic *substitui* o default mutável por uma função geradora. O efeito é o mesmo, mas o
> mecanismo documentado é uma **cópia (deep copy)** do default. A prova de que as listas são independentes:

In [16]:
a, b = Model(), Model()
a.elements.append(1)
print(f"a.elements, b.elements: {a.elements, b.elements!r}")

a.elements, b.elements: ([1], [])


Chegamos às *default factories* e ao porquê de precisarmos delas (seja implicitamente, por defaults mutáveis, seja explicitamente com `Field`).

Digamos que queremos que o default de um campo seja sempre o horário UTC atual.

Se tentarmos isto:

In [17]:
print(f"datetime.now(UTC).isoformat(): {datetime.now(UTC).isoformat()}")

class Log(BaseModel):
    dt: datetime = datetime.now(UTC)
    message: str

datetime.now(UTC).isoformat(): 2026-11-11T11:25:49.824151+00:00


Caímos no mesmo problema das funções: o datetime default é calculado **uma vez**, quando a classe do modelo é criada.

In [18]:
print(f'Log(message="line 1"): {Log(message="line 1")!r}')

Log(message="line 1"): Log(dt=datetime.datetime(2026, 11, 11, 11, 25, 49, 824280, tzinfo=datetime.timezone.utc), message='line 1')


In [19]:
time.sleep(2)
print(f'Log(message="line 2"): {Log(message="line 2")!r}')

Log(message="line 2"): Log(dt=datetime.datetime(2026, 11, 11, 11, 25, 49, 824280, tzinfo=datetime.timezone.utc), message='line 2')


Exatamente o mesmo datetime.

Em vez disso, usamos o argumento `default_factory` do `Field` para indicar uma função que o Pydantic chama a cada criação de instância
que precise de default.

A função não recebe argumentos e deve devolver um valor compatível com o tipo do campo.

In [20]:
from pydantic import Field

class Log(BaseModel):
    dt: datetime = Field(default_factory=lambda: datetime.now(UTC))
    message: str

E agora temos o resultado desejado:

In [21]:
print(f'Log(message="line 1"): {Log(message="line 1")!r}')

Log(message="line 1"): Log(dt=datetime.datetime(2026, 11, 11, 11, 25, 51, 854495, tzinfo=datetime.timezone.utc), message='line 1')


In [22]:
time.sleep(2)
print(f'Log(message="line 2"): {Log(message="line 2")!r}')

Log(message="line 2"): Log(dt=datetime.datetime(2026, 11, 11, 11, 25, 53, 860462, tzinfo=datetime.timezone.utc), message='line 2')


O datetime default foi recalculado nas duas vezes.

Para outros defaults mutáveis, como uma lista vazia, poderíamos usar uma *default factory*:

In [23]:
class Model(BaseModel):
    elements: list[int] = Field(default_factory=lambda: [])

(ou `default_factory=list`, que é a forma idiomática) ou, como o Pydantic permite, simplesmente usar o default mutável direto na classe,
como começamos:

In [24]:
class Model(BaseModel):
    elements: list[int] = []